# Data Access and Storage with Python — Practice Notebook

**Language:** Python 3  
**Suggested duration:** 150–180 minutes  
**Context:** Business analytics, customers, products, sales, and operations  
**Data sources:** CSV, Excel, JSON, HTML, PDF, SQLite, and MongoDB-style documents

This notebook accompanies the lecture **Data Access and Storage with Python**.

Each section follows the same learning pattern:

1. Core concept or command;
2. A working example;
3. An exercise placed immediately after the related content;
4. Hints and partially completed `TODO` code;
5. An integrated practical project near the end.

> Recommended workflow: run the notebook once from top to bottom, then return to the cells marked `TODO`, complete the code, and run those cells again.

## Learning Objectives

After completing this notebook, you should be able to:

- read and write numerical text data with NumPy;
- import CSV files with Pandas using `usecols`, `dtype`, `parse_dates`, `na_values`, and `chunksize`;
- read and write Excel workbooks containing multiple worksheets;
- read JSON and flatten nested JSON structures;
- extract tables from HTML and understand the PDF-table extraction workflow;
- connect to SQLite, execute queries, use parameters, and transfer data between SQL and Pandas;
- work with MongoDB-style documents and convert them to DataFrames;
- integrate multiple sources into a single analytical dataset;
- store outputs in CSV, Excel, and SQLite.

## 0. Environment Setup

If needed, install the supporting packages:

```bash
pip install numpy pandas openpyxl lxml html5lib beautifulsoup4 pdfplumber reportlab pymongo
```

The notebook creates its own sample data files so that no external dataset is required.

In [ ]:
import sys
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path("data_access_storage_lab")
SOURCE_DIR = BASE_DIR / "sources"
OUTPUT_DIR = BASE_DIR / "outputs"

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Working directory:", BASE_DIR.resolve())

## 1. Create the Sample Data Sources

The notebook uses one consistent retail-business scenario.

The setup cell creates:

- `customers.csv`;
- `numeric_sales.csv`;
- `products.xlsx`;
- `orders.json`;
- `market_data.html`;
- `company.db`;
- `large_transactions.csv`.

A small PDF report is also created when `reportlab` is installed.

The source-generation code is complete so that the practice focuses on **data access, validation, integration, and storage** rather than on building raw datasets.

In [ ]:
rng = np.random.default_rng(42)

# customers.csv
customers = pd.DataFrame({
    "CustomerID": ["001", "002", "003", "004", "005", "006", "007", "008"],
    "CustomerName": [
        "An Nguyen", "Binh Tran", "Chi Le", "Dung Pham",
        "Giang Vu", "Ha Do", "Khanh Hoang", "Linh Bui"
    ],
    "City": ["Hanoi", "Hanoi", "Danang", "HCMC", "HCMC", "Danang", "Hanoi", "HCMC"],
    "Age": [28, 35, 31, 42, np.nan, 26, 39, 33],
    "SignupDate": [
        "2025-01-10", "2025-02-18", "2025-03-07", "2025-03-21",
        "2025-04-15", "2025-05-09", "2025-06-01", "2025-06-19"
    ],
    "Segment": ["Retail", "Corporate", "Retail", "SME", "Corporate", "Retail", "SME", "Corporate"],
    "TotalSpent": [1250, 2480, 890, 3150, 2760, 720, 1980, 3520]
})

customers_file = customers.copy().astype({"CustomerID": str})
customers_file["Age"] = customers_file["Age"].astype(object)
customers_file.loc[4, "Age"] = "NA"
customers_file.loc[5, "Segment"] = "-"
customers_file.to_csv(SOURCE_DIR / "customers.csv", index=False)

# numeric_sales.csv
numeric_sales = pd.DataFrame({
    "Month": [1, 2, 3, 4, 5, 6],
    "Revenue": [120, 150, 180, 210, 240, 260],
    "Cost": [80, np.nan, 110, 135, np.nan, 170]
})
numeric_sales.to_csv(SOURCE_DIR / "numeric_sales.csv", index=False)

# products.xlsx
products = pd.DataFrame({
    "ProductID": ["P01", "P02", "P03", "P04", "P05"],
    "ProductName": ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset"],
    "Category": ["Computers", "Accessories", "Accessories", "Accessories", "Accessories"],
    "UnitPrice": [1200, 320, 80, 35, 95]
})

inventory = pd.DataFrame({
    "ProductID": ["P01", "P02", "P03", "P04", "P05"],
    "Stock": [12, 25, 60, 90, 45],
    "ReorderLevel": [5, 10, 20, 30, 15]
})

with pd.ExcelWriter(SOURCE_DIR / "products.xlsx", engine="openpyxl") as writer:
    products.to_excel(writer, sheet_name="Products", index=False)
    inventory.to_excel(writer, sheet_name="Inventory", index=False)

# orders.json
orders = [
    {
        "OrderID": "O001",
        "OrderDate": "2025-07-01",
        "Customer": {"CustomerID": "001", "City": "Hanoi"},
        "Items": [
            {"ProductID": "P01", "Quantity": 1},
            {"ProductID": "P04", "Quantity": 2}
        ]
    },
    {
        "OrderID": "O002",
        "OrderDate": "2025-07-02",
        "Customer": {"CustomerID": "003", "City": "Danang"},
        "Items": [
            {"ProductID": "P02", "Quantity": 2},
            {"ProductID": "P03", "Quantity": 3}
        ]
    },
    {
        "OrderID": "O003",
        "OrderDate": "2025-07-03",
        "Customer": {"CustomerID": "008", "City": "HCMC"},
        "Items": [
            {"ProductID": "P05", "Quantity": 2},
            {"ProductID": "P04", "Quantity": 1}
        ]
    },
    {
        "OrderID": "O004",
        "OrderDate": "2025-07-04",
        "Customer": {"CustomerID": "006", "City": "Danang"},
        "Items": [
            {"ProductID": "P03", "Quantity": 1},
            {"ProductID": "P04", "Quantity": 2}
        ]
    }
]

with open(SOURCE_DIR / "orders.json", "w", encoding="utf-8") as f:
    json.dump(orders, f, ensure_ascii=False, indent=2)

# market_data.html
market_prices = pd.DataFrame({
    "Product": ["Laptop", "Monitor", "Keyboard"],
    "MarketPrice": [1225, 335, 82]
})
fx_rates = pd.DataFrame({
    "Currency": ["USD", "EUR", "JPY"],
    "RateToVND": [25200, 29600, 171]
})

html = (
    "<html><body><h1>Market Data</h1>"
    + market_prices.to_html(index=False)
    + "<h2>FX Rates</h2>"
    + fx_rates.to_html(index=False)
    + "</body></html>"
)
(SOURCE_DIR / "market_data.html").write_text(html, encoding="utf-8")

# company.db
conn_setup = sqlite3.connect(SOURCE_DIR / "company.db")
employees = pd.DataFrame({
    "EmployeeID": [1, 2, 3, 4, 5],
    "EmployeeName": ["Mai", "Nam", "Hoa", "Son", "Trang"],
    "Department": ["Sales", "Sales", "Operations", "Finance", "Operations"],
    "Salary": [28000000, 34000000, 30000000, 42000000, 31500000]
})
employees.to_sql("employees", conn_setup, if_exists="replace", index=False)
customers[["CustomerID", "CustomerName", "City"]].to_sql(
    "customers", conn_setup, if_exists="replace", index=False
)
conn_setup.close()

# large_transactions.csv
n_rows = 20000
large_transactions = pd.DataFrame({
    "TransactionID": np.arange(1, n_rows + 1),
    "CustomerID": rng.choice(customers["CustomerID"], n_rows),
    "Amount": rng.gamma(shape=2.5, scale=80, size=n_rows).round(2),
    "Channel": rng.choice(["Online", "Store", "Partner"], n_rows, p=[0.55, 0.30, 0.15])
})
large_transactions.to_csv(SOURCE_DIR / "large_transactions.csv", index=False)

print("Created source files:")
for path in sorted(SOURCE_DIR.iterdir()):
    print("-", path.name)

### Inspect the Sources Before Analysis

Before using any source, identify:

1. what one row or one document represents;
2. which fields are identifiers, categories, numerical values, or dates;
3. whether any special missing-value markers are used;
4. which fields can serve as merge keys.

In [ ]:
preview = pd.read_csv(SOURCE_DIR / "customers.csv")
print(preview.head().to_string(index=False))
print("\nShape:", preview.shape)
print("\nData types:")
print(preview.dtypes)

# Part 1. Numerical CSV Data with NumPy

## 2. Reading Simple Numerical Data with `np.loadtxt()`

`np.loadtxt()` is appropriate for regular numerical data with no problematic missing values.

In [ ]:
sales_numeric = np.loadtxt(
    SOURCE_DIR / "numeric_sales.csv",
    delimiter=",",
    skiprows=1,
    usecols=[0, 1]
)

print(sales_numeric)
print("Shape:", sales_numeric.shape)
print("Mean revenue:", sales_numeric[:, 1].mean())

### Exercise 1 — `np.loadtxt()`

Read the `Month` and `Revenue` columns from `numeric_sales.csv`.

Then:

1. print the array;
2. print its shape;
3. calculate the maximum revenue.

**Hint:** use `delimiter=","`, `skiprows=1`, and `usecols=[0, 1]`.

In [ ]:
# TODO
# arr_ex1 = np.loadtxt(
#     SOURCE_DIR / "numeric_sales.csv",
#     delimiter=...,
#     skiprows=...,
#     usecols=...
# )
# print(arr_ex1)
# print(arr_ex1.shape)
# print(arr_ex1[:, 1].max())

## 3. Handling Missing Numerical Values with `np.genfromtxt()`

`np.genfromtxt()` is more flexible when numerical files contain missing values.

In [ ]:
sales_with_missing = np.genfromtxt(
    SOURCE_DIR / "numeric_sales.csv",
    delimiter=",",
    skip_header=1,
    filling_values=0.0
)

print(sales_with_missing)

### Exercise 2 — `np.genfromtxt()`

Read the same file and replace missing values with `-1`.

Then inspect the `Cost` column.

**Hint:** use `filling_values=-1`.

In [ ]:
# TODO
# arr_ex2 = np.genfromtxt(
#     SOURCE_DIR / "numeric_sales.csv",
#     delimiter=",",
#     skip_header=1,
#     filling_values=...
# )
# print(arr_ex2[:, 2])

## 4. Writing Numerical Data with `np.savetxt()`

In [ ]:
processed = np.array([
    [1, 120.25, 80.10],
    [2, 150.75, 95.00],
    [3, 180.50, 110.20]
])

np.savetxt(
    OUTPUT_DIR / "processed_numeric_sales.csv",
    processed,
    delimiter=",",
    fmt="%.2f",
    header="Month,Revenue,Cost",
    comments=""
)

print((OUTPUT_DIR / "processed_numeric_sales.csv").read_text())

### Exercise 3 — `np.savetxt()`

Save `sales_with_missing` to `numeric_sales_filled.csv`.

Requirements:

- comma delimiter;
- two decimal places;
- header `Month,Revenue,Cost`;
- no `#` before the header.

In [ ]:
# TODO
# np.savetxt(
#     OUTPUT_DIR / "numeric_sales_filled.csv",
#     sales_with_missing,
#     delimiter=...,
#     fmt=...,
#     header=...,
#     comments=...
# )

# Part 2. CSV Files with Pandas

## 5. Basic CSV Import with `pd.read_csv()`

In [ ]:
customers_raw = pd.read_csv(SOURCE_DIR / "customers.csv")
print(customers_raw.head().to_string(index=False))
print("\nShape:", customers_raw.shape)

### Exercise 4 — Basic CSV Import

Read `customers.csv` into a DataFrame named `customers_ex4`.

Display:

1. the first five rows;
2. the shape;
3. the column names.

In [ ]:
# TODO
# customers_ex4 = pd.read_csv(...)
# print(customers_ex4.head())
# print(customers_ex4.shape)
# print(customers_ex4.columns)

## 6. Reading Selected Columns with `usecols`

Selecting columns during import can reduce memory use and remove unnecessary variables early.

In [ ]:
customer_subset = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    usecols=["CustomerID", "City", "TotalSpent"]
)

print(customer_subset.head().to_string(index=False))

### Exercise 5 — `usecols`

Read only:

- `CustomerID`;
- `CustomerName`;
- `Segment`;
- `TotalSpent`.

Store the result in `customers_ex5`.

In [ ]:
# TODO
# customers_ex5 = pd.read_csv(
#     SOURCE_DIR / "customers.csv",
#     usecols=[...]
# )

## 7. Controlling Data Types and Missing Values

Identifiers such as `CustomerID` should generally be treated as labels rather than quantities.

In [ ]:
customers_typed = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    na_values=["NA", "-"]
)

print(customers_typed.dtypes)
print("\nMissing values:")
print(customers_typed.isna().sum())

### Exercise 6 — `dtype` and `na_values`

Read `customers.csv` so that:

- `CustomerID` is `str`;
- `Age` is `Int64`;
- `NA` and `-` are treated as missing values.

Then inspect `dtypes` and the missing-value counts.

In [ ]:
# TODO
# customers_ex6 = pd.read_csv(
#     SOURCE_DIR / "customers.csv",
#     dtype={...},
#     na_values=[...]
# )
# print(customers_ex6.dtypes)
# print(customers_ex6.isna().sum())

## 8. Parsing Dates with `parse_dates`

In [ ]:
customers_dates = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

print(customers_dates.dtypes)
print(customers_dates[["CustomerID", "SignupDate"]].head().to_string(index=False))

### Exercise 7 — `parse_dates`

Read `customers.csv` and parse `SignupDate` as datetime.

Then print the earliest and latest signup dates.

In [ ]:
# TODO
# customers_ex7 = pd.read_csv(
#     SOURCE_DIR / "customers.csv",
#     dtype={"CustomerID": str},
#     parse_dates=[...],
#     na_values=["NA", "-"]
# )
# print("Earliest:", customers_ex7["SignupDate"].min())
# print("Latest:", customers_ex7["SignupDate"].max())

## 9. Inspecting Data After Import

A useful inspection sequence is:

```python
df.head()
df.shape
df.info()
df.dtypes
df.isna().sum()
```

In [ ]:
customers_clean = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

print(customers_clean.head().to_string(index=False))
print("\nShape:", customers_clean.shape)
print("\nMissing values:")
print(customers_clean.isna().sum())

### Exercise 8 — Data Inspection

Using `customers_clean`, determine:

1. the number of rows;
2. the number of columns;
3. the number of missing values in `Age`;
4. the number of missing values in `Segment`.

In [ ]:
# TODO
# print("Rows:", ...)
# print("Columns:", ...)
# print("Missing Age:", ...)
# print("Missing Segment:", ...)

## 10. Processing a Larger CSV File with `chunksize`

`chunksize` returns an iterator of DataFrames so that a file can be processed incrementally.

In [ ]:
total_amount = 0.0
row_count = 0

for chunk in pd.read_csv(
    SOURCE_DIR / "large_transactions.csv",
    chunksize=5000
):
    total_amount += chunk["Amount"].sum()
    row_count += len(chunk)

print("Rows processed:", row_count)
print("Total amount:", round(total_amount, 2))

### Exercise 9 — `chunksize`

Process `large_transactions.csv` in chunks of 4,000 rows and calculate:

1. total `Amount`;
2. total number of rows;
3. average transaction value.

Do not load the complete file into one DataFrame for the calculation.

In [ ]:
# TODO
# total = 0.0
# n = 0
# for chunk in pd.read_csv(
#     SOURCE_DIR / "large_transactions.csv",
#     chunksize=...
# ):
#     total += ...
#     n += ...
# average = total / n
# print(total, n, average)

## 11. Writing CSV Output

In [ ]:
customers_clean.to_csv(
    OUTPUT_DIR / "customers_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", OUTPUT_DIR / "customers_clean.csv")

### Knowledge Check — CSV

**Question 1.** Which argument reads only selected columns?

A. `usecols`  
B. `columns_only`  
C. `keepcols`  
D. `filter_cols`

**Question 2.** Which argument is directly useful when a CSV file is larger than available RAM?

A. `chunksize`  
B. `index_col`  
C. `header`  
D. `names`

**Question 3.** Why should `CustomerID` often be stored as `str`?

A. It is an identifier rather than a numerical measurement.  
B. It should be averaged.  
C. It is always missing.  
D. Pandas does not support numerical IDs.

# Part 3. Microsoft Excel

## 12. Reading a Single Worksheet

In [ ]:
products_df = pd.read_excel(
    SOURCE_DIR / "products.xlsx",
    sheet_name="Products",
    engine="openpyxl"
)

print(products_df.to_string(index=False))

### Exercise 10 — `read_excel()`

Read the `Inventory` worksheet into `inventory_ex10` and display the first five rows.

In [ ]:
# TODO
# inventory_ex10 = pd.read_excel(
#     SOURCE_DIR / "products.xlsx",
#     sheet_name=...,
#     engine="openpyxl"
# )
# print(inventory_ex10.head())

## 13. Reading All Worksheets

Using `sheet_name=None` returns a dictionary of DataFrames.

In [ ]:
workbook = pd.read_excel(
    SOURCE_DIR / "products.xlsx",
    sheet_name=None,
    engine="openpyxl"
)

print("Worksheets:", list(workbook.keys()))

for sheet_name, sheet_df in workbook.items():
    print(sheet_name, sheet_df.shape)

### Exercise 11 — Multiple Worksheets

Using `workbook`:

1. access `Products`;
2. access `Inventory`;
3. merge them on `ProductID`.

In [ ]:
# TODO
# products_ex11 = workbook[...]
# inventory_ex11 = workbook[...]
# product_inventory = products_ex11.merge(...)
# print(product_inventory)

## 14. Writing Multiple Worksheets

In [ ]:
inventory_df = workbook["Inventory"]
product_inventory = products_df.merge(
    inventory_df,
    on="ProductID",
    how="left"
)

with pd.ExcelWriter(
    OUTPUT_DIR / "product_report.xlsx",
    engine="openpyxl"
) as writer:
    products_df.to_excel(writer, sheet_name="Products", index=False)
    inventory_df.to_excel(writer, sheet_name="Inventory", index=False)
    product_inventory.to_excel(writer, sheet_name="Combined", index=False)

print("Saved:", OUTPUT_DIR / "product_report.xlsx")

### Exercise 12 — Excel Export

Create `inventory_report.xlsx` with two worksheets:

- `Inventory`;
- `Low_Stock`.

Define low stock as:

```text
Stock <= ReorderLevel
```

In [ ]:
# TODO
# low_stock = inventory_df[
#     inventory_df["Stock"] <= inventory_df["ReorderLevel"]
# ]
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "inventory_report.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

# Part 4. JSON

## 15. Reading JSON

In [ ]:
with open(
    SOURCE_DIR / "orders.json",
    "r",
    encoding="utf-8"
) as f:
    orders_data = json.load(f)

print("Number of orders:", len(orders_data))
print(json.dumps(orders_data[0], indent=2))

### Exercise 13 — Inspect JSON Structure

For the first order:

1. print `OrderID`;
2. print the customer ID;
3. print the number of items.

In [ ]:
# TODO
# first_order = orders_data[0]
# print(first_order[...])
# print(first_order["Customer"][...])
# print(len(first_order[...]))

## 16. Flattening Nested JSON with `pd.json_normalize()`

Each item in the nested `Items` list becomes one row.

In [ ]:
order_items = pd.json_normalize(
    orders_data,
    record_path=["Items"],
    meta=[
        "OrderID",
        "OrderDate",
        ["Customer", "CustomerID"],
        ["Customer", "City"]
    ]
)

order_items = order_items.rename(columns={
    "Customer.CustomerID": "CustomerID",
    "Customer.City": "City"
})

print(order_items.to_string(index=False))

### Exercise 14 — `pd.json_normalize()`

Create `items_ex14` with:

- one row per item;
- `OrderID`;
- `OrderDate`;
- `CustomerID`;
- `City`;
- `ProductID`;
- `Quantity`.

Then verify that its number of rows equals the total number of items in all orders.

In [ ]:
# TODO
# items_ex14 = pd.json_normalize(
#     orders_data,
#     record_path=[...],
#     meta=[...]
# )
# total_items = sum(len(order["Items"]) for order in orders_data)
# print("Rows:", len(items_ex14))
# print("Expected:", total_items)

### Knowledge Check — JSON

**Question 1.** What does `record_path` identify?

A. The nested list to expand into rows  
B. The output filename  
C. The database name  
D. A missing-value marker

**Question 2.** What does `meta` preserve?

A. Parent-level information  
B. Only numerical fields  
C. File permissions  
D. HTML tags

# Part 5. HTML and PDF

## 17. Reading HTML Tables with `pd.read_html()`

In [ ]:
html_tables = pd.read_html(SOURCE_DIR / "market_data.html")

print("Number of HTML tables:", len(html_tables))

for i, table in enumerate(html_tables):
    print(f"\nTable {i}")
    print(table.to_string(index=False))

### Exercise 15 — HTML Tables

Using `html_tables`:

1. store the first table as `market_prices_ex15`;
2. store the second table as `fx_rates_ex15`;
3. print their shapes;
4. export both tables to separate worksheets in `market_tables.xlsx`.

In [ ]:
# TODO
# market_prices_ex15 = ...
# fx_rates_ex15 = ...
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "market_tables.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

## 18. PDF Table Extraction

PDF is primarily a presentation format. A table that appears visually structured may not be stored internally as an explicit table.

The next cell creates a small PDF report when `reportlab` is available.

In [ ]:
pdf_path = SOURCE_DIR / "sales_report.pdf"

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
    from reportlab.lib.styles import getSampleStyleSheet

    doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
    styles = getSampleStyleSheet()

    pdf_data = [
        ["Region", "Revenue", "Orders"],
        ["Hanoi", "4200", "38"],
        ["Danang", "2750", "24"],
        ["HCMC", "5100", "44"]
    ]

    table = Table(pdf_data)
    table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("ALIGN", (1, 1), (-1, -1), "RIGHT")
    ]))

    doc.build([
        Paragraph("Regional Sales Report", styles["Title"]),
        Spacer(1, 12),
        table
    ])

    print("Created:", pdf_path)

except ImportError:
    print("reportlab is not installed. PDF generation skipped.")

In [ ]:
try:
    import pdfplumber

    if pdf_path.exists():
        with pdfplumber.open(pdf_path) as pdf:
            first_page = pdf.pages[0]
            extracted_tables = first_page.extract_tables()

        print("Extracted tables:", len(extracted_tables))

        if extracted_tables:
            raw_table = extracted_tables[0]
            pdf_df = pd.DataFrame(raw_table[1:], columns=raw_table[0])
            print(pdf_df.to_string(index=False))

except ImportError:
    print("pdfplumber is not installed. Install it with: pip install pdfplumber")

### Exercise 16 — Inspect Extracted PDF Data

If `pdf_df` exists:

1. inspect `head()`, `shape`, `columns`, missing values, and data types;
2. convert `Revenue` and `Orders` to numerical types.

In [ ]:
# TODO
# if "pdf_df" in globals():
#     print(pdf_df.head())
#     print(pdf_df.shape)
#     print(pdf_df.columns)
#     print(pdf_df.isna().sum())
#     print(pdf_df.dtypes)
#
#     pdf_df["Revenue"] = pd.to_numeric(pdf_df["Revenue"], errors="coerce")
#     pdf_df["Orders"] = pd.to_numeric(pdf_df["Orders"], errors="coerce")

# Part 6. SQLite

## 19. Connecting to SQLite

In [ ]:
conn = sqlite3.connect(SOURCE_DIR / "company.db")
print("Connection opened.")

## 20. Reading SQL Results into Pandas

In [ ]:
employee_df = pd.read_sql_query(
    "SELECT EmployeeID, EmployeeName, Department, Salary FROM employees",
    conn
)

print(employee_df.to_string(index=False))

### Exercise 17 — SQL Filtering

Write a query that returns only employees in the `Sales` department.

Load the result into `sales_employees`.

In [ ]:
# TODO
# sales_employees = pd.read_sql_query(
#     "SELECT ... FROM employees WHERE ...",
#     conn
# )
# print(sales_employees)

## 21. Parameterized Queries

A parameterized query separates SQL structure from supplied values.

In [ ]:
salary_threshold = 30000000

high_salary = pd.read_sql_query(
    "SELECT EmployeeID, EmployeeName, Department, Salary FROM employees WHERE Salary >= ?",
    conn,
    params=(salary_threshold,)
)

print(high_salary.to_string(index=False))

### Exercise 18 — Parameterized SQL

Set:

```python
department_name = "Operations"
```

Use a parameterized query to retrieve employees from that department.

In [ ]:
# TODO
# department_name = "Operations"
# operations_staff = pd.read_sql_query(
#     "SELECT ... FROM employees WHERE Department = ?",
#     conn,
#     params=(department_name,)
# )
# print(operations_staff)

## 22. Writing a DataFrame to SQLite

In [ ]:
city_summary = (
    customers_clean
    .groupby("City", as_index=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        TotalSpent=("TotalSpent", "sum")
    )
)

city_summary.to_sql(
    "city_customer_summary",
    conn,
    if_exists="replace",
    index=False
)

print(
    pd.read_sql_query(
        "SELECT * FROM city_customer_summary",
        conn
    ).to_string(index=False)
)

### Exercise 19 — `to_sql()`

Create a table named `segment_customer_summary` containing:

- Segment;
- number of customers;
- total spending.

Use `if_exists="replace"`.

In [ ]:
# TODO
# segment_summary = (
#     customers_clean
#     .groupby("Segment", as_index=False)
#     .agg(
#         Customers=("CustomerID", "nunique"),
#         TotalSpent=("TotalSpent", "sum")
#     )
# )
#
# segment_summary.to_sql(
#     "segment_customer_summary",
#     conn,
#     if_exists="replace",
#     index=False
# )

### Knowledge Check — SQLite

**Question 1.** Which function opens an SQLite connection?

A. `sqlite3.connect()`  
B. `pd.connect_sqlite()`  
C. `sqlite.open()`  
D. `pd.read_db()`

**Question 2.** Which Pandas function loads query results into a DataFrame?

A. `pd.read_sql_query()`  
B. `pd.sql_to_frame()`  
C. `pd.read_database_file()`  
D. `pd.load_sql()`

**Question 3.** Why should parameterized queries be preferred to direct string concatenation?

A. They separate SQL structure from supplied values and reduce SQL injection risk.  
B. They remove all missing values.  
C. They eliminate the need for a database connection.  
D. They automatically build indexes.

# Part 7. MongoDB-Style Documents

## 23. Document-Oriented Data

A MongoDB collection contains documents rather than relational rows. Documents can contain nested dictionaries and lists.

The first example uses ordinary Python dictionaries so that the notebook runs without a MongoDB server.

In [ ]:
mongo_style_orders = [
    {
        "_id": "m001",
        "OrderID": "M001",
        "Customer": {"CustomerID": "001", "City": "Hanoi"},
        "TotalAmount": 1500,
        "Status": "PAID"
    },
    {
        "_id": "m002",
        "OrderID": "M002",
        "Customer": {"CustomerID": "003", "City": "Danang"},
        "TotalAmount": 820,
        "Status": "SHIPPED"
    },
    {
        "_id": "m003",
        "OrderID": "M003",
        "Customer": {"CustomerID": "008", "City": "HCMC"},
        "TotalAmount": 2100,
        "Status": "PAID"
    }
]

mongo_df = pd.json_normalize(mongo_style_orders)
print(mongo_df.to_string(index=False))

### Exercise 20 — MongoDB-Style Documents

Using `mongo_style_orders`:

1. keep only documents with `Status == "PAID"`;
2. convert them to a DataFrame;
3. calculate total `TotalAmount`.

In [ ]:
# TODO
# paid_docs = [
#     doc for doc in mongo_style_orders
#     if doc["Status"] == "PAID"
# ]
# paid_df = pd.json_normalize(...)
# print(...)

## 24. Optional Live MongoDB Connection

This section is optional. It runs only when:

- `pymongo` is installed;
- a MongoDB server is available at `mongodb://localhost:27017/`.

A short timeout prevents the notebook from hanging when no server is available.

In [ ]:
try:
    from pymongo import MongoClient

    client = MongoClient(
        "mongodb://localhost:27017/",
        serverSelectionTimeoutMS=1000
    )

    client.admin.command("ping")

    db = client["retail_lab"]
    collection = db["orders"]

    collection.delete_many({})
    collection.insert_many(mongo_style_orders)

    live_documents = list(collection.find({"Status": "PAID"}))
    live_df = pd.json_normalize(live_documents)

    if "_id" in live_df.columns:
        live_df["_id"] = live_df["_id"].astype(str)

    print(live_df.to_string(index=False))

except Exception as exc:
    print("Live MongoDB example skipped:", type(exc).__name__)

# Part 8. Integrated Multi-Source Data Pipeline

## 25. Business Scenario

Build an analytical order dataset by integrating:

- customer information from CSV;
- product information from Excel;
- order items from nested JSON.

The final table should contain:

- OrderID;
- OrderDate;
- CustomerID;
- CustomerName;
- City;
- ProductID;
- ProductName;
- Category;
- Quantity;
- UnitPrice;
- Revenue.

## 26. Step 1 — Load and Inspect the Sources

In [ ]:
customers_pipeline = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

products_pipeline = pd.read_excel(
    SOURCE_DIR / "products.xlsx",
    sheet_name="Products"
)

with open(SOURCE_DIR / "orders.json", encoding="utf-8") as f:
    orders_pipeline = json.load(f)

items_pipeline = pd.json_normalize(
    orders_pipeline,
    record_path=["Items"],
    meta=[
        "OrderID",
        "OrderDate",
        ["Customer", "CustomerID"],
        ["Customer", "City"]
    ]
).rename(columns={
    "Customer.CustomerID": "CustomerID",
    "Customer.City": "OrderCity"
})

items_pipeline["OrderDate"] = pd.to_datetime(items_pipeline["OrderDate"])

print("Customers:", customers_pipeline.shape)
print("Products:", products_pipeline.shape)
print("Order items:", items_pipeline.shape)

### Exercise 21 — Validate Keys Before Merging

Check:

1. whether `CustomerID` is unique in the customer table;
2. whether `ProductID` is unique in the product table;
3. whether any `ProductID` in the order items is missing from the product table.

In [ ]:
# TODO
# print("CustomerID unique:", customers_pipeline["CustomerID"].is_unique)
# print("ProductID unique:", products_pipeline["ProductID"].is_unique)
#
# missing_product_ids = (
#     set(items_pipeline["ProductID"])
#     - set(products_pipeline["ProductID"])
# )
# print("Missing Product IDs:", missing_product_ids)

## 27. Step 2 — Merge the Sources

In [ ]:
order_analytics = (
    items_pipeline
    .merge(
        products_pipeline,
        on="ProductID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        customers_pipeline[
            ["CustomerID", "CustomerName", "City", "Segment"]
        ],
        on="CustomerID",
        how="left",
        validate="many_to_one"
    )
)

order_analytics["Revenue"] = (
    order_analytics["Quantity"]
    * order_analytics["UnitPrice"]
)

print(order_analytics.head().to_string(index=False))
print("\nMissing values after merge:")
print(order_analytics.isna().sum())

### Exercise 22 — Post-Merge Validation

Using `order_analytics`:

1. compare `OrderCity` and `City`;
2. count rows where they differ;
3. verify that `Revenue` contains no missing values;
4. calculate total Revenue.

In [ ]:
# TODO
# city_mismatch = order_analytics["OrderCity"] != order_analytics["City"]
# print("City mismatches:", city_mismatch.sum())
# print("Missing Revenue:", order_analytics["Revenue"].isna().sum())
# print("Total Revenue:", order_analytics["Revenue"].sum())

## 28. Step 3 — Create KPI Tables

In [ ]:
city_kpi = (
    order_analytics
    .groupby("City", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("OrderID", "nunique"),
        Customers=("CustomerID", "nunique")
    )
)

category_kpi = (
    order_analytics
    .groupby("Category", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum")
    )
)

print("City KPI")
print(city_kpi.to_string(index=False))

print("\nCategory KPI")
print(category_kpi.to_string(index=False))

## 29. Step 4 — Store the Analytical Output

In [ ]:
analytics_conn = sqlite3.connect(OUTPUT_DIR / "analytics.db")

order_analytics.to_sql(
    "order_analytics",
    analytics_conn,
    if_exists="replace",
    index=False
)
city_kpi.to_sql(
    "city_kpi",
    analytics_conn,
    if_exists="replace",
    index=False
)
category_kpi.to_sql(
    "category_kpi",
    analytics_conn,
    if_exists="replace",
    index=False
)

with pd.ExcelWriter(
    OUTPUT_DIR / "analytics_dashboard.xlsx",
    engine="openpyxl"
) as writer:
    order_analytics.to_excel(writer, sheet_name="Details", index=False)
    city_kpi.to_excel(writer, sheet_name="City_KPI", index=False)
    category_kpi.to_excel(writer, sheet_name="Category_KPI", index=False)

analytics_conn.close()

print("Created:", OUTPUT_DIR / "analytics.db")
print("Created:", OUTPUT_DIR / "analytics_dashboard.xlsx")

### Exercise 23 — Integrated Pipeline Extension

Create `segment_kpi` containing:

- Segment;
- number of unique customers;
- number of unique orders;
- total Revenue.

Store it:

1. in `analytics.db` as table `segment_kpi`;
2. in `segment_report.xlsx` as worksheet `Segment_KPI`.

In [ ]:
# TODO
# segment_kpi = (
#     order_analytics
#     .groupby(...)
#     .agg(...)
#     .reset_index()
# )
#
# conn2 = sqlite3.connect(OUTPUT_DIR / "analytics.db")
# segment_kpi.to_sql("segment_kpi", conn2, if_exists="replace", index=False)
# conn2.close()
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "segment_report.xlsx",
#     engine="openpyxl"
# ) as writer:
#     segment_kpi.to_excel(writer, sheet_name="Segment_KPI", index=False)

# Part 9. Best Practices

## 30. A Reusable Inspection Pattern

A practical workflow is:

```text
Source
   ↓
Read
   ↓
Inspect
   ↓
Validate
   ↓
Transform
   ↓
Integrate
   ↓
Analyze
   ↓
Store
```

Useful checks include:

- `head()`;
- `shape`;
- `info()`;
- `dtypes`;
- `isna().sum()`;
- key uniqueness;
- unmatched merge keys;
- range and category checks.

In [ ]:
print("Rows:", len(order_analytics))
print("Columns:", len(order_analytics.columns))
print("Unique orders:", order_analytics["OrderID"].nunique())
print("Unique customers:", order_analytics["CustomerID"].nunique())
print("Missing ProductName:", order_analytics["ProductName"].isna().sum())
print("Missing CustomerName:", order_analytics["CustomerName"].isna().sum())

### Exercise 24 — Storage Format Selection

Choose an appropriate format for each situation and explain your choice.

1. Send a simple table to a colleague.
2. Produce a multi-sheet management report.
3. Exchange nested data through a Web API.
4. Store transactional data that must support SQL queries.
5. Store flexible nested documents.
6. Store a large analytical table efficiently for repeated use.

Possible formats include CSV, Excel, JSON, SQLite, MongoDB, and Parquet.

# Part 10. Worked Practical Exercise

## 31. Customer Spending Summary

### Task

Using `customers.csv`:

1. read IDs as strings;
2. parse `SignupDate`;
3. interpret `NA` and `-` as missing values;
4. calculate total and average spending by City;
5. store the result in CSV and SQLite.

The following cell provides a complete reference solution.

In [ ]:
customers_worked = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

customer_city_summary = (
    customers_worked
    .groupby("City", as_index=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        TotalSpent=("TotalSpent", "sum"),
        AverageSpent=("TotalSpent", "mean")
    )
)

customer_city_summary["AverageSpent"] = (
    customer_city_summary["AverageSpent"].round(2)
)

customer_city_summary.to_csv(
    OUTPUT_DIR / "customer_city_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

conn_ref = sqlite3.connect(OUTPUT_DIR / "customer_summary.db")
customer_city_summary.to_sql(
    "customer_city_summary",
    conn_ref,
    if_exists="replace",
    index=False
)
conn_ref.close()

print(customer_city_summary.to_string(index=False))

# Part 11. Final Independent Project

## 32. Retail Data Access and Storage Project

You are a data analyst preparing a reusable analytical dataset for management.

### Required Sources

Use:

- `customers.csv`;
- `products.xlsx`;
- `orders.json`;
- `company.db`.

### Required Tasks

1. Read all relevant sources using appropriate options.
2. Inspect shapes, data types, missing values, and key uniqueness.
3. Flatten the nested JSON order data.
4. Merge customers, products, and order items.
5. Calculate line-level Revenue.
6. Create at least three KPI tables.
7. Run at least one parameterized SQLite query.
8. Store the detailed analytical dataset in SQLite.
9. Export the KPI tables to a multi-sheet Excel workbook.
10. Export at least one final table to CSV using Excel-friendly UTF-8 encoding.

### Validation Requirements

The final dataset should satisfy:

- `CustomerID` remains a string;
- `OrderDate` is datetime before export;
- every order item has a valid product;
- every order item has a valid customer;
- `Revenue = Quantity × UnitPrice`;
- core analytical fields contain no missing values.

In [ ]:
# TODO — FINAL PROJECT WORKSPACE

# 1. Read sources
# ...

# 2. Inspect and validate
# ...

# 3. Flatten JSON
# ...

# 4. Merge
# ...

# 5. Create Revenue
# ...

# 6. Build KPI tables
# ...

# 7. Query SQLite
# ...

# 8. Store outputs
# ...

print("Complete the final project in this cell or add new cells below.")

## 33. Optional Self-Checks

After completing the final project, adapt these assertions to your variable names:

```python
assert final_df["CustomerID"].dtype == object
assert pd.api.types.is_datetime64_any_dtype(final_df["OrderDate"])
assert final_df["CustomerName"].notna().all()
assert final_df["ProductName"].notna().all()
assert final_df["Revenue"].notna().all()

assert np.allclose(
    final_df["Revenue"],
    final_df["Quantity"] * final_df["UnitPrice"]
)
```

Assertions convert important data-quality assumptions into executable checks.

## 34. Data Access and Storage Cheat Sheet

| Goal | Main command |
|---|---|
| Read simple numerical text | `np.loadtxt(...)` |
| Read numerical text with missing values | `np.genfromtxt(...)` |
| Write NumPy text data | `np.savetxt(...)` |
| Read CSV | `pd.read_csv(...)` |
| Read selected CSV columns | `usecols=[...]` |
| Specify types | `dtype={...}` |
| Parse dates | `parse_dates=[...]` |
| Define missing markers | `na_values=[...]` |
| Read large CSV incrementally | `chunksize=...` |
| Write CSV | `df.to_csv(...)` |
| Read Excel | `pd.read_excel(...)` |
| Read all Excel worksheets | `sheet_name=None` |
| Write multiple worksheets | `pd.ExcelWriter(...)` |
| Read JSON | `json.load(...)`, `pd.read_json(...)` |
| Flatten nested JSON | `pd.json_normalize(...)` |
| Read HTML tables | `pd.read_html(...)` |
| Extract PDF tables | `pdfplumber` |
| Connect to SQLite | `sqlite3.connect(...)` |
| Read SQL into Pandas | `pd.read_sql_query(...)` |
| Write Pandas to SQL | `df.to_sql(...)` |
| Normalize MongoDB documents | `pd.json_normalize(...)` |

### Submission Checklist

- [ ] Appropriate import options are used.
- [ ] IDs use appropriate data types.
- [ ] Dates are parsed correctly.
- [ ] Missing values are inspected.
- [ ] Merge keys are validated.
- [ ] Post-merge missing values are checked.
- [ ] Large data is processed appropriately.
- [ ] SQL queries use parameters when values come from variables.
- [ ] Output formats match the business requirement.
- [ ] The pipeline can be rerun from the beginning.

## Conclusion

The notebook follows the complete workflow:

```text
Create / Obtain Sources
        ↓
Read
        ↓
Inspect
        ↓
Validate
        ↓
Transform
        ↓
Integrate
        ↓
Analyze
        ↓
Store / Report
```

The next step is to replace the simulated sources with real datasets while preserving the same data-access, validation, integration, and storage workflow.

In [ ]:
try:
    conn.close()
except Exception:
    pass

print("Core practice notebook completed.")
print("Outputs are stored in:", OUTPUT_DIR.resolve())